# Minh họa pipeline dữ liệu hiện tại

Notebook này dùng để trình bày nhanh pipeline đang phục vụ Dashboard. `scripts/clean_data.py` mới là nơi tạo các file đầu ra chính; notebook chỉ đọc và hiển thị, không ghi file.

## Luồng xử lý

`data/raw/` → `scripts/clean_data.py` → `data/processed/` → Dashboard

Pipeline tạo ba nhóm dữ liệu: bán hàng/RFM, Walmart và BigMart.

In [10]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'data').is_dir())
RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'

raw_files = pd.DataFrame({'File raw': sorted(p.name for p in RAW.glob('*.csv'))})
raw_files

,File raw
0,bigmart_train.csv
1,global_electronics_retail_2019_2020.csv
2,global_superstore_orders.csv
3,global_superstore_people.csv
4,superstore_2015_2018.csv
5,superstore_2021_2024.csv
6,walmart_features.csv
7,walmart_stores.csv
8,walmart_train.csv


## Các bảng sau pipeline

Bảng bán hàng dùng cho Tổng quan, Địa lý, RFM và Dự báo. Bảng Walmart và BigMart được giữ riêng vì có cấu trúc và mục đích khác.

In [11]:
tables = {
    'Bán hàng': 'orders_enriched.csv',
    'RFM': 'rfm_customers.csv',
    'Doanh thu tháng': 'monthly_sales.csv',
    'Walmart': 'walmart_weekly_enriched.csv',
    'BigMart': 'bigmart_cleaned.csv',
}

summary = []
for name, filename in tables.items():
    frame = pd.read_csv(PROCESSED / filename, nrows=5, low_memory=False)
    summary.append({'Nhóm dữ liệu': name, 'File': filename, 'Số cột': len(frame.columns), 'Mẫu dữ liệu': frame.head(2)})
pd.DataFrame(summary)[['Nhóm dữ liệu', 'File', 'Số cột']]

,Nhóm dữ liệu,File,Số cột
0,Bán hàng,orders_enriched.csv,12
1,RFM,rfm_customers.csv,13
2,Doanh thu tháng,monthly_sales.csv,6
3,Walmart,walmart_weekly_enriched.csv,16
4,BigMart,bigmart_cleaned.csv,13


## Ví dụ dữ liệu sau khi làm sạch

Bảng dưới cho thấy dữ liệu bán hàng đã có các cột chuẩn như ngày, khách hàng, khu vực, danh mục, doanh thu, số lượng, lợi nhuận và nguồn dữ liệu.

In [12]:
orders = pd.read_csv(PROCESSED / 'orders_enriched.csv', nrows=8, parse_dates=['Order Date'])
orders

,Order ID,Order Date,Customer ID,Country,Region,Category,Sub-Category,Sales,Quantity,Profit,Data Source,Manager
0,AG-2012-TB112803-40909,2012-01-01,TB-112803,Algeria,North Africa,Office Supplies,Storage,408.300,2,106.140,Global Superstore (2012-2015),Lindiwe Afolayan
1,IN-2012-JH159857-40909,2012-01-01,JH-159857,Australia,Oceania,Office Supplies,Supplies,120.366,3,36.036,Global Superstore (2012-2015),Kauri Anaru
2,IN-2012-JH159857-40909,2012-01-01,JH-159857,Australia,Oceania,Furniture,Furnishings,113.670,5,37.770,Global Superstore (2012-2015),Kauri Anaru
3,IT-2012-EM14140124-40909,2012-01-01,EM-14140124,Sweden,Northern Europe,Office Supplies,Paper,44.865,3,-26.055,Global Superstore (2012-2015),Miina Nylund
4,IN-2012-JH159857-40909,2012-01-01,JH-159857,Australia,Oceania,Office Supplies,Paper,55.242,2,15.342,Global Superstore (2012-2015),Kauri Anaru
5,HU-2012-AT73557-40909,2012-01-01,AT-73557,Hungary,Eastern Europe,Office Supplies,Storage,66.120,4,29.640,Global Superstore (2012-2015),Oxana Lagunov
6,CA-2012-MM726023-40910,2012-01-02,MM-726023,Canada,Canada,Technology,Machines,314.220,1,3.120,Global Superstore (2012-2015),NaN
7,IZ-2012-LW699061-40911,2012-01-03,LW-699061,Iraq,Western Asia,Office Supplies,Envelopes,47.430,1,17.070,Global Superstore (2012-2015),Kaoru Xun


## Minh họa kết quả pipeline

Biểu đồ này cho thấy quy mô dòng dữ liệu của các nhóm chính. Đây là phần minh họa để trình bày, không tạo thêm file.

In [13]:
row_counts = pd.DataFrame({
    'Nhóm dữ liệu': list(tables),
    'Số dòng mẫu hiển thị': [len(pd.read_csv(PROCESSED / f, nrows=1000, low_memory=False)) for f in tables.values()]
})
px.bar(row_counts, x='Nhóm dữ liệu', y='Số dòng mẫu hiển thị', title='Các nhóm dữ liệu sau pipeline')